
# Classical QSPR Pipeline  
## Linear Regression + Validation + Visualization + Regression Equations

This notebook implements a complete classical QSPR workflow including:

- Descriptor selection using correlation analysis
- Linear regression modeling
- Leave-One-Out Cross Validation (LOO-Q²)
- Observed vs Predicted plots
- Residual plots
- Williams plots
- Export of regression equations and validation metrics

The workflow is designed for publication-quality QSPR studies.


In [ ]:
# ================================================================
# CLASSICAL QSPR PIPELINE
# Linear
# Validation + Visualization + Regression Equations (Q1 Ready)
# ================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score

# ================================================================
# 1. PATHS
# ================================================================
base_path = r"File path"

corr_df = pd.read_excel(base_path + r"\Best correlations.xlsx", index_col=0)
data_df = pd.read_excel(base_path + r"\complete data.xlsx")

corr_df.index = corr_df.index.astype(str).str.strip()
data_df.columns = data_df.columns.astype(str).str.strip()

# ================================================================
# 2. FUNCTIONS
# ================================================================
def loo_q2(model, X, y):
    loo = LeaveOneOut()
    preds = []

    for tr, ts in loo.split(X):
        model.fit(X[tr], y[tr])
        preds.append(model.predict(X[ts])[0])

    preds = np.array(preds)

    return 1 - np.sum((y - preds)**2) / np.sum((y - np.mean(y))**2)

def leverage(X):
    H = X @ np.linalg.pinv(X.T @ X) @ X.T
    return np.diagonal(H)

# ================================================================
# 3. TARGET PROPERTIES
# ================================================================
properties = ["MW", "Complexity"]

results = []

# ================================================================
# 4. MODELING + VISUALIZATION
# ================================================================
for prop in properties:

    # ------------------------------------------------------------
    # Best Descriptor Selection
    # ------------------------------------------------------------
    desc = corr_df[prop].abs().idxmax()

    X = data_df[[desc]].values
    y = data_df[prop].values

    mask = ~np.isnan(X).flatten() & ~np.isnan(y)

    X = X[mask]
    y = y[mask]

    # ------------------------------------------------------------
    # Linear Regression Model
    # ------------------------------------------------------------
    model = LinearRegression()
    model.fit(X, y)

    y_pred = model.predict(X)

    r2 = r2_score(y, y_pred)
    q2 = loo_q2(model, X, y)

    # ------------------------------------------------------------
    # Regression Equation
    # ------------------------------------------------------------
    intercept = model.intercept_
    coefficient = model.coef_[0]

    equation = f"{prop} = {intercept:.6f} + ({coefficient:.6f}) × {desc}"

    print("\n================================================")
    print(f"Property      : {prop}")
    print(f"Descriptor    : {desc}")
    print(f"Regression Eq : {equation}")
    print(f"R²            : {r2:.4f}")
    print(f"Q²_LOO        : {q2:.4f}")
    print("================================================")

    # ------------------------------------------------------------
    # Observed vs Predicted Plot
    # ------------------------------------------------------------
    plt.figure()

    plt.scatter(y, y_pred)

    plt.plot(
        [y.min(), y.max()],
        [y.min(), y.max()],
        '--'
    )

    plt.xlabel("Observed")
    plt.ylabel("Predicted")
    plt.title(f"{prop}: Observed vs Predicted")

    plt.savefig(
        base_path + fr"\{prop}_Obs_vs_Pred.png",
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

    # ------------------------------------------------------------
    # Residuals Plot
    # ------------------------------------------------------------
    plt.figure()

    plt.scatter(y, y - y_pred)

    plt.axhline(0, linestyle='--')

    plt.xlabel("Observed")
    plt.ylabel("Residuals")
    plt.title(f"{prop}: Residuals vs Observed")

    plt.savefig(
        base_path + fr"\{prop}_Residuals.png",
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

    # ------------------------------------------------------------
    # Williams Plot
    # ------------------------------------------------------------
    h = leverage(X)

    h_star = 3 * (X.shape[1] + 1) / X.shape[0]

    std_res = (y - y_pred) / np.std(y - y_pred)

    plt.figure()

    plt.scatter(h, std_res)

    plt.axhline(3, linestyle='--')
    plt.axhline(-3, linestyle='--')
    plt.axvline(h_star, linestyle='--')

    plt.xlabel("Leverage")
    plt.ylabel("Standardized Residuals")
    plt.title(f"{prop}: Williams Plot")

    plt.savefig(
        base_path + fr"\{prop}_Williams.png",
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

    # ------------------------------------------------------------
    # Save Results
    # ------------------------------------------------------------
    results.append([
        prop,
        desc,
        equation,
        r2,
        q2
    ])

# ================================================================
# 5. SAVE RESULTS
# ================================================================
results_df = pd.DataFrame(
    results,
    columns=[
        "Property",
        "Descriptor",
        "Regression_Equation",
        "R2",
        "Q2_LOO"
    ]
)

results_df.to_excel(
    base_path + r"\Classical_QSPR_Final.xlsx",
    index=False
)

print("\n✔ Classical QSPR + Regression Equations + Visuals Completed")



## Output Files

The notebook automatically generates:

- Regression equations
- Validation metrics (R² and Q²_LOO)
- Observed vs Predicted plots
- Residual plots
- Williams plots
- Final Excel summary file

Make sure to update the `base_path` before execution.
